In [ ]:
from google.colab import files
files.upload()  # kaggle.json

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

!pip install -q kaggle
!kaggle datasets download -d blastchar/telco-customer-churn
!unzip -o telco-customer-churn.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
License(s): copyright-authors
100% 172k/172k [00:00<00:00, 480kB/s]

Archive:  telco-customer-churn.zip
  inflating: WA_Fn-UseC_-Telco-Customer-Churn.csv  


In [ ]:
import pandas as pd

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.isnull().sum())
print(df['Churn'].value_counts())

(7043, 21)
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object
customerID          0
gender              0
SeniorCitizen      

In [ ]:
# Fix TotalCharges — blanks become NaN, then convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# These blank rows are customers with tenure = 0 (just signed up, no charge yet)
# Fill with 0 rather than dropping — they're valid customers, not bad data
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Drop CustomerID duplicates if any
df = df.drop_duplicates(subset='customerID')

# Convert target to binary for easier aggregation
df['Churn_Flag'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
# Tenure cohort buckets — this is your cohort analysis backbone
df['Tenure Cohort'] = pd.cut(
    df['tenure'],
    bins=[-1, 6, 12, 24, 48, 100],
    labels=['0-6 mo', '7-12 mo', '13-24 mo', '25-48 mo', '48+ mo']
)

# Monthly-to-total charge ratio — flags customers paying disproportionately
df['Avg Charge Per Month'] = df['TotalCharges'] / df['tenure'].replace(0, 1)

# Simplify contract type for cohort readability (already categorical, but confirm)
# Contract: Month-to-month, One year, Two year

# Services subscribed count — proxy for "engagement depth"
service_cols = ['PhoneService','MultipleLines','InternetService','OnlineSecurity',
                 'OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
df['Services Count'] = df[service_cols].apply(
    lambda row: sum(1 for v in row if v not in ['No', 'No internet service', 'No phone service']),
    axis=1
)

In [ ]:
cohort_summary = df.groupby('Tenure Cohort').agg(
    Total_Customers=('customerID', 'count'),
    Churned=('Churn_Flag', 'sum'),
    Avg_Monthly_Charge=('MonthlyCharges', 'mean')
).reset_index()
cohort_summary['Churn Rate %'] = (cohort_summary['Churned'] / cohort_summary['Total_Customers']) * 100
cohort_summary.to_csv('cohort_summary.csv', index=False)

/tmp/ipykernel_864/620137384.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cohort_summary = df.groupby('Tenure Cohort').agg(


In [ ]:
df.to_csv('cleaned_telco_churn.csv', index=False)

from google.colab import files
files.download('cleaned_telco_churn.csv')
files.download('cohort_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>